# Multimodal Confusion Pattern Analysis

Minimal notebook to analyze confusion matrices across model variants.

- Assumes this notebook lives in `analysis/multimodal/`
- Assumes prediction files live in `outputs_multimodal/*/Dy*/organoid_predictions.csv`
- Focus: model/backbone/input/fusion, **not** day.

## 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

## 2. Load predictions and parse model metadata

In [ ]:
ROOT = Path("outputs_multimodal")

model_dirs = [
    d for d in ROOT.iterdir()
    if d.is_dir() and d.name not in {"overall", "logs"}
]

print("Found model directories:")
for d in model_dirs:
    print(" -", d.name)

all_preds = []

for model_dir in model_dirs:
    model_id = model_dir.name
    for day_dir in model_dir.iterdir():
        if not day_dir.is_dir():
            continue
        csv_path = day_dir / "organoid_predictions.csv"
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        df["Model_ID"] = model_id
        all_preds.append(df)

if not all_preds:
    raise RuntimeError("No organoid_predictions.csv files found under outputs_multimodal.")

preds = pd.concat(all_preds, ignore_index=True)
print("Total prediction rows:", len(preds))

In [ ]:
def parse_model_id(mid: str):
    parts = mid.split("_")
    backbone = parts[0]
    if len(parts) == 3:
        input_mode = parts[1]
        fusion = parts[2]
    elif len(parts) == 4:
        input_mode = parts[1] + "_" + parts[2]
        fusion = parts[3]
    else:
        input_mode = "unknown"
        fusion = "unknown"
    return backbone, input_mode, fusion

parsed = preds["Model_ID"].apply(parse_model_id)
preds["Backbone"] = parsed.apply(lambda x: x[0])
preds["Input_Mode"] = parsed.apply(lambda x: x[1])
preds["Fusion"] = parsed.apply(lambda x: x[2])

preds["Cell_ID"] = preds["Organoid_ID"].astype(str)

print("Backbones:", preds["Backbone"].unique())
print("Input modes:", preds["Input_Mode"].unique())
print("Fusion strategies:", preds["Fusion"].unique())
print("Unique cells:", preds["Cell_ID"].nunique())

## 3. Confusion statistics by model and by backbone/input/fusion

In [ ]:
cm = (
    preds.groupby("Model_ID")["CM_Category"]
    .value_counts()
    .unstack(fill_value=0)
)
cm = cm.reindex(columns=["TP", "FP", "TN", "FN"], fill_value=0)
model_cm = cm.reset_index()

model_cm["accuracy"] = (model_cm["TP"] + model_cm["TN"]) / (
    model_cm["TP"] + model_cm["TN"] + model_cm["FP"] + model_cm["FN"]
)
model_cm["precision"] = model_cm["TP"] / (model_cm["TP"] + model_cm["FP"]).replace(0, np.nan)
model_cm["recall"] = model_cm["TP"] / (model_cm["TP"] + model_cm["FN"]).replace(0, np.nan)
model_cm["specificity"] = model_cm["TN"] / (model_cm["TN"] + model_cm["FP"]).replace(0, np.nan)

print("\nPer-model confusion summary (sorted by accuracy):")
display(model_cm.sort_values("accuracy", ascending=False))

## 4. Cell-level consistency across all models

In [ ]:
cell_cm = (
    preds.groupby("Cell_ID")
    .agg(
        n_runs=("Model_ID", "count"),
        TP=("CM_Category", lambda s: (s == "TP").sum()),
        FP=("CM_Category", lambda s: (s == "FP").sum()),
        TN=("CM_Category", lambda s: (s == "TN").sum()),
        FN=("CM_Category", lambda s: (s == "FN").sum()),
    )
)

cell_cm["frac_TP"] = cell_cm["TP"] / cell_cm["n_runs"]
cell_cm["frac_FP"] = cell_cm["FP"] / cell_cm["n_runs"]
cell_cm["frac_TN"] = cell_cm["TN"] / cell_cm["n_runs"]
cell_cm["frac_FN"] = cell_cm["FN"] / cell_cm["n_runs"]

print("Total cells:", len(cell_cm))

In [ ]:
EASY_THRESH = 0.90
HARD_THRESH = 0.50

easy_cells = cell_cm[cell_cm["frac_TP"] >= EASY_THRESH]
hard_FN_cells = cell_cm[cell_cm["frac_FN"] >= HARD_THRESH]
hard_FP_cells = cell_cm[cell_cm["frac_FP"] >= HARD_THRESH]

print("Easy cells (mostly TP):", len(easy_cells))
print("Hard FN cells (often missed):", len(hard_FN_cells))
print("Hard FP cells (often false alarms):", len(hard_FP_cells))

## 5. Inspect example hard and easy cells

In [ ]:
print("Top hard FN cells:")
display(hard_FN_cells.sort_values("frac_FN", ascending=False).head(10))

print("Top hard FP cells:")
display(hard_FP_cells.sort_values("frac_FP", ascending=False).head(10))

print("Sample easy cells:")
display(easy_cells.sort_values("frac_TP", ascending=False).head(10))

## efficientnet_rgb_mask_concat

In [ ]:
eff_mask_concat = preds[preds["Model_ID"] == "efficientnet_rgb_mask_concat"]

eff_cell = (
    eff_mask_concat.groupby("Cell_ID")
    .agg(
        n_runs=("Model_ID", "count"),
        TP=("CM_Category", lambda s: (s=="TP").sum()),
        FP=("CM_Category", lambda s: (s=="FP").sum()),
        TN=("CM_Category", lambda s: (s=="TN").sum()),
        FN=("CM_Category", lambda s: (s=="FN").sum()),
    )
)

eff_cell["frac_TP"] = eff_cell["TP"] / eff_cell["n_runs"]
eff_cell["frac_FP"] = eff_cell["FP"] / eff_cell["n_runs"]
eff_cell["frac_TN"] = eff_cell["TN"] / eff_cell["n_runs"]
eff_cell["frac_FN"] = eff_cell["FN"] / eff_cell["n_runs"]

hard_FN_eff = eff_cell[eff_cell["frac_FN"] > 0]
hard_FP_eff = eff_cell[eff_cell["frac_FP"] > 0]
easy_eff = eff_cell[eff_cell["frac_TP"] == 1]

print("Top hard FN cells for efficientnet_rgb_mask_concat:")
display(hard_FN_eff.sort_values("frac_FN", ascending=False).head(10))

print("Top hard FP cells for efficientnet_rgb_mask_concat:")
display(hard_FP_eff.sort_values("frac_FP", ascending=False).head(10))

print("Sample easy cells for efficientnet_rgb_mask_concat:")
display(easy_eff.sort_values("frac_TP", ascending=False).head(10))